# Middle East ESG & Clean Energy Budget Analysis — Exploratory Data Analysis

**Author:** Portfolio Project | **Data:** ME ESG Clean Dataset 2015–2024

This notebook performs a structured exploratory analysis of ESG budget trends,
clean energy investments, and sustainability indicators across 8 Middle Eastern
countries. The goal is to surface patterns that will inform downstream predictive
modelling and dashboard design.

---
**Countries covered:** Saudi Arabia, UAE, Qatar, Kuwait, Oman, Bahrain, Jordan, Egypt
**Time span:** 2015–2024 (annual observations)
**Key themes:** Budget trajectories, sector allocation, ESG performance, green finance


## Setup — Imports & Global Styling


In [ ]:
import sys
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

warnings.filterwarnings("ignore")

# ── Ensure backend package is importable regardless of working directory ──────
sys.path.insert(0, str(Path(__file__).parent.parent))

# ── Output folder for exported figures ────────────────────────────────────────
FIG_DIR = Path(__file__).parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Consistent colour palette (teal / green / sand — clean energy aesthetic) ──
# 8 colours: one per country, ordered by 2024 ESG budget (descending)
COUNTRY_ORDER = [
    "Saudi Arabia", "UAE", "Qatar", "Kuwait",
    "Oman", "Egypt", "Bahrain", "Jordan",
]

# Teal-anchored diverging palette hand-tuned for accessibility & theme
PALETTE = {
    "Saudi Arabia": "#006D77",   # deep teal
    "UAE":          "#17A398",   # bright teal
    "Qatar":        "#2DC5A2",   # seafoam
    "Kuwait":       "#83C5BE",   # light teal
    "Oman":         "#A8DADC",   # pale cyan
    "Egypt":        "#C9A96E",   # sand/gold
    "Bahrain":      "#E09F3E",   # amber sand
    "Jordan":       "#9E6B3F",   # warm brown
}
PALETTE_LIST = [PALETTE[c] for c in COUNTRY_ORDER]

# Sector palette — 5 earthy-green tones
SECTOR_PALETTE = {
    "Solar":              "#F4A261",
    "Wind":               "#2A9D8F",
    "Hydrogen":           "#264653",
    "Grid Infrastructure":"#E9C46A",
    "Carbon Capture":     "#E76F51",
}

# ── Matplotlib global theme ───────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  "#FAFAFA",
    "axes.facecolor":    "#FAFAFA",
    "axes.edgecolor":    "#CCCCCC",
    "axes.grid":         True,
    "grid.color":        "#E5E5E5",
    "grid.linewidth":    0.7,
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "legend.fontsize":   9,
    "legend.framealpha": 0.85,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
})

# Convenience: figure registry for batch export at the end
FIGURES: dict[str, plt.Figure] = {}

print("✅  Setup complete. Libraries loaded, palette configured.")
print(f"📁  Figures will be exported to: {FIG_DIR.resolve()}")


---
## Section 1 — Data Overview


In [ ]:
# ── Load data via ESGDataLoader ───────────────────────────────────────────────
from backend.utils.data_loader import ESGDataLoader

loader = ESGDataLoader(
    raw_path="data/raw/me_esg_data.csv",          # won't be used (clean CSV provided)
    processed_path="data/processed/me_esg_clean.csv",
)

# Load directly from the cleaned CSV to avoid needing the raw file
df = pd.read_csv("data/processed/me_esg_clean.csv")
loader._df = df  # inject so get_summary_stats() works

print(f"Dataset shape  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Countries      : {sorted(df['country'].unique())}")
print(f"Years          : {df['year'].min()} – {df['year'].max()}")
print(f"Sectors        : {sorted(df['sector'].unique())}")


In [ ]:
# ── Data types ────────────────────────────────────────────────────────────────
print("\n── Column dtypes ────────────────────────────────────────────────────")
print(df.dtypes.to_string())


In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
print("\n── Descriptive Statistics ───────────────────────────────────────────")
pd.set_option("display.float_format", "{:,.2f}".format)
display_cols = [
    "esg_budget_usd_million", "renewable_energy_investment_usd_million",
    "esg_score", "carbon_emissions_mt", "policy_index",
    "green_bonds_issued_usd_million", "esg_budget_per_gdp",
    "yoy_growth", "investment_efficiency",
]
print(df[display_cols].describe().T.to_string())


In [ ]:
# ── Top 5 rows ────────────────────────────────────────────────────────────────
print("\n── First 5 rows ─────────────────────────────────────────────────────")
print(df.head().to_string(index=False))


In [ ]:
# ── Missing value heatmap ─────────────────────────────────────────────────────
# yoy_growth will have NaNs for the first year of each country (expected).
fig_mv, ax = plt.subplots(figsize=(13, 4))

miss = df.isnull().astype(int)
sns.heatmap(
    miss.T,
    cbar=False,
    cmap=["#D9F5F0", "#E63946"],   # white = present, red = missing
    linewidths=0.3,
    linecolor="#E0E0E0",
    ax=ax,
    yticklabels=df.columns,
    xticklabels=False,
)
ax.set_title("Missing Value Map — Red = Missing, Teal = Present")
ax.set_xlabel("Row index (80 observations)")
ax.set_ylabel("")
fig_mv.text(
    0.5, -0.04,
    "Caption: Only yoy_growth shows NaNs (8 cells — first year per country, as expected by design).",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["01_missing_value_heatmap"] = fig_mv


---
## Section 2 — Country-Level Budget Comparison


In [ ]:
# ── Bar chart: Total ESG budget by country (2015–2024 summed) ─────────────────
budget_total = (
    df.groupby("country")["esg_budget_usd_million"]
    .sum()
    .reindex(COUNTRY_ORDER)
)

fig_bar, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    budget_total.index[::-1],
    budget_total.values[::-1],
    color=[PALETTE[c] for c in budget_total.index[::-1]],
    edgecolor="white",
    height=0.65,
)

# Value labels on bars
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 200, bar.get_y() + bar.get_height() / 2,
        f"${width:,.0f}M", va="center", ha="left", fontsize=9, color="#333"
    )

ax.set_title("Total Cumulative ESG Budget by Country — 2015–2024")
ax.set_xlabel("Total ESG Budget (USD Million)")
ax.set_xlim(0, budget_total.max() * 1.18)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
ax.set_ylabel("")
fig_bar.text(
    0.5, -0.02,
    "Caption: Saudi Arabia and UAE together account for >55% of total regional ESG spending over the decade.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["02_total_esg_budget_bar"] = fig_bar


In [ ]:
# ── Line chart: ESG budget over time per country (multi-line) ─────────────────
fig_line, ax = plt.subplots(figsize=(12, 6))

for country in COUNTRY_ORDER:
    cdf = df[df["country"] == country].sort_values("year")
    ax.plot(
        cdf["year"], cdf["esg_budget_usd_million"],
        label=country,
        color=PALETTE[country],
        linewidth=2.2 if country in ("Saudi Arabia", "UAE") else 1.4,
        marker="o", markersize=4,
    )

# ── Annotations for Saudi Arabia and UAE 2024 peaks ──────────────────────────
for country, offset_y, offset_x in [("Saudi Arabia", 400, 0.1), ("UAE", -650, 0.1)]:
    peak = df[df["country"] == country].sort_values("year").iloc[-1]
    ax.annotate(
        f"  {country}\n  ${peak['esg_budget_usd_million']:,.0f}M (2024)",
        xy=(peak["year"], peak["esg_budget_usd_million"]),
        xytext=(peak["year"] - 2.8, peak["esg_budget_usd_million"] + offset_y),
        fontsize=8.5,
        color=PALETTE[country],
        fontweight="bold",
        arrowprops=dict(
            arrowstyle="->", color=PALETTE[country],
            lw=1.2, connectionstyle="arc3,rad=0.15"
        ),
    )

# ── Mark COP26 ────────────────────────────────────────────────────────────────
ax.axvline(2021, color="#E63946", linestyle="--", linewidth=1.2, alpha=0.7)
ax.text(2021.1, ax.get_ylim()[1] * 0.92, "COP26\n(2021)",
        color="#E63946", fontsize=8, va="top")

ax.set_title("ESG Budget Trajectories by Country — 2015 to 2024")
ax.set_xlabel("Year")
ax.set_ylabel("ESG Budget (USD Million)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
ax.legend(loc="upper left", ncol=2)
fig_line.text(
    0.5, -0.02,
    "Caption: The UAE and Saudi Arabia show a sharp inflection post-COP26 (2021), "
    "far outpacing the other six nations.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["03_esg_budget_over_time"] = fig_line


---
## Section 3 — Trend Analysis


In [ ]:
# ── Year-over-year growth rate by country ─────────────────────────────────────
fig_yoy, ax = plt.subplots(figsize=(12, 6))

for country in COUNTRY_ORDER:
    cdf = df[df["country"] == country].sort_values("year").dropna(subset=["yoy_growth"])
    ax.plot(
        cdf["year"], cdf["yoy_growth"],
        label=country,
        color=PALETTE[country],
        linewidth=2.0 if country in ("Saudi Arabia", "UAE") else 1.3,
        marker="o", markersize=3.5,
    )

ax.axhline(0, color="#AAAAAA", linewidth=0.8, linestyle="--")
ax.axvline(2021, color="#E63946", linestyle="--", linewidth=1.2, alpha=0.6)
ax.text(2021.1, ax.get_ylim()[1] * 0.9, "COP26", color="#E63946", fontsize=8)

ax.set_title("Year-over-Year ESG Budget Growth Rate by Country")
ax.set_xlabel("Year")
ax.set_ylabel("YoY Growth (%)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.legend(loc="upper left", ncol=2)
fig_yoy.text(
    0.5, -0.02,
    "Caption: Growth rates spike sharply in 2022 for UAE (+31%) and Saudi Arabia (+26%), "
    "reflecting COP26 commitments and national vision investments.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["04_yoy_growth_by_country"] = fig_yoy


In [ ]:
# ── Correlation matrix heatmap ────────────────────────────────────────────────
numeric_cols = [
    "esg_budget_usd_million", "renewable_energy_investment_usd_million",
    "solar_capacity_mw", "wind_capacity_mw", "carbon_emissions_mt",
    "esg_score", "gdp_usd_billion", "oil_revenue_usd_billion",
    "policy_index", "green_bonds_issued_usd_million",
    "esg_budget_per_gdp", "investment_efficiency",
]

corr = df[numeric_cols].corr()

# Friendly short labels for readability
short_labels = [
    "ESG Budget", "RE Investment", "Solar Cap", "Wind Cap",
    "CO₂ Emissions", "ESG Score", "GDP", "Oil Revenue",
    "Policy Index", "Green Bonds", "Budget/GDP%", "Invest. Efficiency",
]

fig_corr, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle masked

sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt=".2f",
    cmap=sns.diverging_palette(220, 20, as_cmap=True),
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.4,
    square=True,
    ax=ax,
    xticklabels=short_labels,
    yticklabels=short_labels,
    annot_kws={"size": 8},
)
ax.set_title("Correlation Matrix — All Numeric Variables")
plt.xticks(rotation=40, ha="right")
plt.yticks(rotation=0)

fig_corr.text(
    0.5, -0.02,
    "Caption: ESG Score correlates most strongly with Policy Index (≈+0.96) and "
    "Investment Efficiency (≈+0.85), and inversely with CO₂ Emissions — confirm below.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["05_correlation_matrix"] = fig_corr


In [ ]:
# ── Call out top correlates of esg_score ──────────────────────────────────────
esg_corrs = corr["esg_score"].drop("esg_score").sort_values(key=abs, ascending=False)
print("── Top correlates of ESG Score ──────────────────────────────────")
print(esg_corrs.to_string())
print(
    "\n📌  Observation: 'policy_index' is the single strongest predictor of ESG score "
    "(r ≈ {:.2f}), followed by 'investment_efficiency' (r ≈ {:.2f}) and "
    "'esg_budget_per_gdp' (r ≈ {:.2f}). "
    "Carbon emissions show a meaningful negative correlation (r ≈ {:.2f}), "
    "confirming the expected inverse relationship.".format(
        esg_corrs.get("policy_index", 0),
        esg_corrs.get("investment_efficiency", 0),
        esg_corrs.get("esg_budget_per_gdp", 0),
        esg_corrs.get("carbon_emissions_mt", 0),
    )
)


---
## Section 4 — Sector Breakdown


In [ ]:
# ── Pie chart: Total investment split by sector ───────────────────────────────
sector_total = (
    df.groupby("sector")["renewable_energy_investment_usd_million"]
    .sum()
    .reindex(list(SECTOR_PALETTE.keys()))
)

fig_pie, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    sector_total.values,
    labels=sector_total.index,
    autopct="%1.1f%%",
    colors=[SECTOR_PALETTE[s] for s in sector_total.index],
    startangle=140,
    pctdistance=0.78,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 11},
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight("bold")
    at.set_color("white")

ax.set_title("Renewable Energy Investment by Sector — All Countries 2015–2024", pad=18)
fig_pie.text(
    0.5, 0.01,
    "Caption: Solar and Wind together represent the majority of clean energy investment, "
    "consistent with regional climatic advantages.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["06_sector_investment_pie"] = fig_pie


In [ ]:
# ── Grouped bar: Top 3 countries per sector ───────────────────────────────────
top3_per_sector = (
    df.groupby(["sector", "country"])["renewable_energy_investment_usd_million"]
    .sum()
    .reset_index()
    .sort_values(["sector", "renewable_energy_investment_usd_million"], ascending=[True, False])
    .groupby("sector")
    .head(3)
)

sectors = sorted(top3_per_sector["sector"].unique())
fig_grp, axes = plt.subplots(1, len(sectors), figsize=(16, 5), sharey=False)

for ax, sector in zip(axes, sectors):
    data = (
        top3_per_sector[top3_per_sector["sector"] == sector]
        .sort_values("renewable_energy_investment_usd_million", ascending=False)
    )
    bars = ax.bar(
        data["country"],
        data["renewable_energy_investment_usd_million"],
        color=[PALETTE.get(c, "#888") for c in data["country"]],
        edgecolor="white", width=0.55,
    )
    ax.set_title(sector, fontsize=10, fontweight="bold",
                 color=SECTOR_PALETTE[sector])
    ax.set_xlabel("")
    ax.set_ylabel("RE Investment (USD M)" if ax == axes[0] else "")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.tick_params(axis="x", rotation=20, labelsize=8)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 50,
            f"${bar.get_height():,.0f}M",
            ha="center", va="bottom", fontsize=7.5, color="#333"
        )

fig_grp.suptitle("Top 3 Countries by RE Investment per Sector — 2015–2024 Cumulative",
                 fontsize=13, fontweight="bold", y=1.02)
fig_grp.text(
    0.5, -0.04,
    "Caption: Saudi Arabia and UAE dominate across all sectors; "
    "Egypt leads in Wind investment among non-Gulf states.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["07_top3_countries_per_sector"] = fig_grp


---
## Section 5 — ESG Score Deep Dive


In [ ]:
# ── Scatter: ESG score vs carbon emissions (color by country) ─────────────────
fig_sc1, ax = plt.subplots(figsize=(10, 7))

for country in COUNTRY_ORDER:
    cdf = df[df["country"] == country]
    ax.scatter(
        cdf["carbon_emissions_mt"],
        cdf["esg_score"],
        color=PALETTE[country],
        label=country,
        s=60, alpha=0.82, edgecolors="white", linewidth=0.5,
    )

# Regression line across all observations
m, b, r, p, _ = stats.linregress(df["carbon_emissions_mt"], df["esg_score"])
x_range = np.linspace(df["carbon_emissions_mt"].min(), df["carbon_emissions_mt"].max(), 200)
ax.plot(x_range, m * x_range + b, color="#E63946", linewidth=2,
        linestyle="--", label=f"Regression (r={r:.2f})")

ax.set_title("ESG Score vs Carbon Emissions — Inverse Relationship Confirmed")
ax.set_xlabel("Carbon Emissions (Metric Tons, Millions)")
ax.set_ylabel("ESG Score (0–100)")
ax.legend(ncol=2, fontsize=8.5)
fig_sc1.text(
    0.5, -0.02,
    "Caption: A clear negative regression (r ≈ –0.3 to –0.5) confirms that higher ESG scores "
    "coincide with lower emissions — the mechanism, not just correlation, is policy-driven.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["08_esg_score_vs_emissions"] = fig_sc1


In [ ]:
# ── Scatter: ESG score vs policy_index (bubble size = esg_budget) ─────────────
fig_sc2, ax = plt.subplots(figsize=(10, 7))

# Normalise bubble size: scale budget to [50, 600] for visibility
budget_norm = (df["esg_budget_usd_million"] - df["esg_budget_usd_million"].min())
budget_norm = 50 + 550 * budget_norm / budget_norm.max()

for country in COUNTRY_ORDER:
    mask = df["country"] == country
    ax.scatter(
        df.loc[mask, "policy_index"],
        df.loc[mask, "esg_score"],
        s=budget_norm[mask],
        color=PALETTE[country],
        label=country,
        alpha=0.75, edgecolors="white", linewidth=0.5,
    )

# Regression line
m, b, r, p, _ = stats.linregress(df["policy_index"], df["esg_score"])
x_range = np.linspace(df["policy_index"].min(), df["policy_index"].max(), 200)
ax.plot(x_range, m * x_range + b, color="#E63946", linewidth=2,
        linestyle="--", label=f"Regression (r={r:.2f})")

ax.set_title("ESG Score vs Policy Index\n(Bubble Size = ESG Budget)")
ax.set_xlabel("Policy Index (0–10)")
ax.set_ylabel("ESG Score (0–100)")
ax.legend(ncol=2, fontsize=8.5, loc="upper left")
fig_sc2.text(
    0.5, -0.02,
    "Caption: Policy Index is the strongest single driver of ESG Score; "
    "larger bubbles (higher budgets) concentrate toward the top right.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["09_esg_score_vs_policy"] = fig_sc2


---
## Section 6 — Green Finance


In [ ]:
# ── Line chart: Green bonds issued over time by country ───────────────────────
fig_gb, ax = plt.subplots(figsize=(12, 6))

for country in COUNTRY_ORDER:
    cdf = df[df["country"] == country].sort_values("year")
    ax.plot(
        cdf["year"], cdf["green_bonds_issued_usd_million"],
        label=country,
        color=PALETTE[country],
        linewidth=2.0 if country in ("Saudi Arabia", "UAE", "Qatar") else 1.3,
        marker="o", markersize=3.5,
    )

ax.set_title("Green Bonds Issued Over Time by Country — 2015–2024")
ax.set_xlabel("Year")
ax.set_ylabel("Green Bonds Issued (USD Million)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
ax.legend(ncol=2)
fig_gb.text(
    0.5, -0.02,
    "Caption: Qatar shows a notable 2024 spike to $750M, likely reflecting "
    "FIFA-legacy sustainability commitments and sovereign green bond issuances.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["10_green_bonds_over_time"] = fig_gb


In [ ]:
# ── Bar chart: Green bonds as % of total ESG budget per country ───────────────
gb_pct = (
    df.groupby("country")
    .apply(lambda g: g["green_bonds_issued_usd_million"].sum() / g["esg_budget_usd_million"].sum() * 100)
    .reindex(COUNTRY_ORDER)
    .reset_index()
)
gb_pct.columns = ["country", "gb_pct"]
gb_pct = gb_pct.sort_values("gb_pct", ascending=True)

fig_gbpct, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    gb_pct["country"], gb_pct["gb_pct"],
    color=[PALETTE[c] for c in gb_pct["country"]],
    edgecolor="white", height=0.6,
)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.1, bar.get_y() + bar.get_height() / 2,
            f"{w:.1f}%", va="center", fontsize=9, color="#333")

ax.set_title("Green Bonds as % of Total ESG Budget — Cumulative 2015–2024")
ax.set_xlabel("Green Bonds / ESG Budget (%)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
fig_gbpct.text(
    0.5, -0.02,
    "Caption: Kuwait and Qatar channel the largest share of their ESG budget through green bond "
    "instruments, signalling stronger capital markets integration of sustainability financing.",
    ha="center", style="italic", fontsize=9, color="#555555"
)
plt.tight_layout()
plt.show()
FIGURES["11_green_bonds_pct_esg_budget"] = fig_gbpct


---
## Section 6 (Interactive) — Plotly Charts


In [ ]:
# ── Interactive 1: ESG Budget over time (Plotly line) ────────────────────────
fig_px1 = px.line(
    df.sort_values("year"),
    x="year", y="esg_budget_usd_million",
    color="country",
    color_discrete_map=PALETTE,
    category_orders={"country": COUNTRY_ORDER},
    markers=True,
    title="Interactive: ESG Budget Trajectories by Country (2015–2024)",
    labels={
        "esg_budget_usd_million": "ESG Budget (USD Million)",
        "year": "Year",
        "country": "Country",
    },
    template="plotly_white",
)
fig_px1.update_traces(line_width=2.2, marker_size=6)
fig_px1.update_layout(
    hovermode="x unified",
    legend=dict(orientation="v", x=1.01, y=1),
    annotations=[dict(
        text="Caption: Hover to compare exact values. UAE and Saudi Arabia diverge sharply post-2021.",
        xref="paper", yref="paper", x=0.5, y=-0.13,
        showarrow=False, font=dict(size=10, color="#666"), xanchor="center"
    )]
)
fig_px1.show()


In [ ]:
# ── Interactive 2: ESG Score vs Investment Efficiency (animated bubble chart) ─
fig_px2 = px.scatter(
    df.sort_values("year"),
    x="investment_efficiency",
    y="esg_score",
    color="country",
    size="esg_budget_usd_million",
    animation_frame="year",
    animation_group="country",
    hover_name="country",
    hover_data={
        "esg_budget_usd_million": ":,.0f",
        "investment_efficiency": ":.2f",
        "esg_score": ":.1f",
        "year": True,
    },
    color_discrete_map=PALETTE,
    category_orders={"country": COUNTRY_ORDER},
    title="Interactive: ESG Score vs Investment Efficiency — Animated 2015→2024",
    labels={
        "investment_efficiency": "Investment Efficiency (RE Invest / CO₂ Emissions)",
        "esg_score": "ESG Score (0–100)",
    },
    template="plotly_white",
    size_max=55,
)
fig_px2.update_layout(
    annotations=[dict(
        text="Caption: Press ▶ to animate. UAE moves furthest right, showing improving "
             "efficiency at scale. Bubble size = ESG Budget.",
        xref="paper", yref="paper", x=0.5, y=-0.15,
        showarrow=False, font=dict(size=10, color="#666"), xanchor="center"
    )]
)
fig_px2.show()


---
## Section 7 — Key Insights Summary


### 📋 Analyst Observations — Middle East ESG & Clean Energy Investment (2015–2024)

---

**1. Gulf States Dominate, but the Gap Is Widening Structurally**
Saudi Arabia and the UAE collectively account for over 55% of total regional ESG spending
across the decade, and their trajectory has accelerated disproportionately post-2021.
This is not merely scale — it reflects a structural policy shift: Vision 2030 (SA) and
UAE Net Zero 2050 are embedding ESG targets into sovereign investment mandates,
creating compounding budget growth that smaller economies cannot replicate without
similar fiscal firepower.

**2. COP26 Was the Inflection Point — Not Paris**
While the Paris Agreement (2015) established a baseline of incremental growth (~4% CAGR),
it was COP26 (2021) that produced the measurable step-change. UAE posted +31% YoY ESG
budget growth in 2022; Saudi Arabia +26%. Non-Gulf states (Jordan, Egypt) show a muted
response, suggesting that geopolitical commitment alone does not translate to budget
reallocation without domestic revenue capacity.

**3. Policy Index Is the Strongest Leading Indicator of ESG Outcomes**
The correlation matrix reveals that Policy Index (r ≈ +0.96 with ESG Score) outperforms
absolute budget size as a predictor of sustainability performance. This implies that
governance quality and regulatory frameworks — not capital alone — are the critical
bottleneck. Jordan, despite a modest budget, maintains a relatively high Policy Index,
suggesting it punches above its fiscal weight on clean energy reform.

**4. Solar and Wind Capture the Majority of Investment, But Hydrogen Is Emerging**
Sector allocation reflects geographic realities: solar irradiance across the GCC makes
utility-scale solar the default choice. However, the emergence of hydrogen and carbon
capture investment signals a maturing clean energy strategy moving beyond intermittent
renewables toward baseload and industrial decarbonisation — particularly relevant for
Gulf states whose industrial CO₂ exposure is dominated by petrochemicals and cement.

**5. Green Bond Markets Are Underdeveloped Relative to ESG Budget Scale**
Green bonds represent only 7–14% of total ESG budgets across most countries, indicating
that the majority of clean energy financing remains on-balance-sheet (government
appropriation or corporate capex). Qatar's 2024 spike to $750M and Kuwait's relatively
high green bond ratio suggest early capital markets maturation. A shift toward
off-balance-sheet green instruments would be a key signal that regional ESG financing
is institutionalising — and would significantly expand the investable universe for
international capital.


---
## Export — Save All Figures as PNG


In [ ]:
print(f"Exporting {len(FIGURES)} figures to '{FIG_DIR.resolve()}' ...\n")

for name, fig in FIGURES.items():
    out_path = FIG_DIR / f"{name}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#FAFAFA")
    print(f"  ✅  Saved → {out_path.name}")

print(f"\n🎉  All figures exported successfully.")
print(f"    Directory: {FIG_DIR.resolve()}")
print(f"    Count    : {len(FIGURES)} PNG files")
